In [14]:
import openai, json
client = openai.OpenAI()
messages =[]

In [15]:
def get_weather(city):
    return "33 degrees celcius."

FUNCTION_MAP = {"get_weather": get_weather}


In [16]:
TOOLS = [ 
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "The function to get the weather of a city.",
            "parameters": {
                "type": "object",
                "properties": {
                "city": {
                    "type": "string",
                    "desription": "The name of the city to get the weather of."
                    }
                },  
                "required": ["city"]
            }
        }
    }
]

In [22]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls > 0:
        message.append(
            {
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [ 
                {
                  "id": tool_call.id,
                  "type": "function",
                  "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments,
                  },
                }
                for tool_call in message.tool_calls
            ],
        }
    )

    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = tool_call.function.arguments

        print(f"Calling function: {function_name} with {arguments}")

        try:
            arguments = json.loads(arguments)
        except json.JSONDecodeError:
            arguments = {}

        function_to_run = FUNCTION_MAP.get(function_name)

        result = function_to_run(**arguments)

        message.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result,
            }
        )




In [21]:
def call_ai():
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS
    )
    process_ai_response(response.choices[0].message)
    print(response)
    message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": message})
    print(f"AI: {message}")

In [23]:
while True:
    message = input("You: ")
    if message == "exit":
        break
    messages.append({"role": "user", "content": message})
    print(f"User: {message}")
    call_ai()
    


User: hi


TypeError: '>' not supported between instances of 'list' and 'int'